# 15 - Evaluate a Trained Model

Load the FrozenLake DQN checkpoint and tokenizer from
[02 — Offline DQN](02_train_offline_dqn.ipynb), then evaluate on new maps with cached
inference. The model weights stay fixed throughout evaluation.

The final cells optionally render replay videos. The loading and cached inference
pattern also applies to other environments.


In [ ]:
import os

import gymnasium as gym
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv
from torch import nn

from mouse_core import load_model, load_tokenizer
from mouse_core.data import EventLog, Tokenizer
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    action_readouts,
    prepare_decode,
    select_discrete_action,
)

# Repositories
# Hugging Face model repo for load_model
MODEL_ID = "mouse-example-model-offline"

# Environment
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20
# number of environment streams in the GroupEnv
NUM_ENVS = 2

# Model — match the implementation and settings used in example 02
PRETRAINED = "Qwen/Qwen3-0.6B"
# maximum previous-token distance; match training and inference
MAX_ATTENTION_DISTANCE_TOKENS = 16383
MAX_ACTIONS = 4
HEAD_LAYERS = 1
HEAD_SCALE = 0.1

# Compute text — fixed while an episode is cached
INFERENCE_COMPUTE_TOKENS = 8

# Hugging Face continuous batching
# maximum new tokens processed in one decoder forward
DECODE_MAX_BATCH_TOKENS = 1024
# shared KV capacity across streams, including 16-token page rounding
DECODE_CACHE_CAPACITY_TOKENS = 32768

# Evaluation
# lockstep env steps for inference (passed to run_eval)
EVAL_STEPS = 512
# used only for video score formatting / frame cutoff
EPISODES_PER_ENV = 1

# Random seeds
# held-out env seed stream (far from train)
EVAL_SEED_OFFSET = 1000000

# Runtime
ensure_registered()
# Render videos without requiring a display or audio device.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Build Environment

The model was trained on rows containing `action`, `observation`, `reward`,
`inner_terminated`, and `inner_truncated`. `_terminated` and `_truncated` are objective
columns. The evaluation environment must emit the same fields and record the zero-based
`repeat_episode_index` only on inner reset events, including the first reset. Rendering
is optional and only used here to create replay videos.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode="rgb_array",
            emit_q_star=False,
        )
        env = RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
        return env

    return env_fn


configs = [
    EnvConfig(
        name=f"proc_frozenlake_{i}",
        seed=i + EVAL_SEED_OFFSET,
        seed_mode="stream",
        env_fn=make_lake(stream=i + EVAL_SEED_OFFSET),
        reset_options={"regenerate_map": True},
    )
    for i in range(NUM_ENVS)
]

eval_env = make_group_env(configs)

logs = [EventLog(name=name) for name in eval_env.names]
trackers = [RepeatTaskTracker() for _ in logs]

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.

Reconstruct example 02's encoder, Q head, and architecture settings before `load_model`;
weights do not reconstruct Python modules. Load the tokenizer from the same bundle. PPO
and GRPO checkpoints need their own policy-head architecture and action selection.

Define `episode_separator` and `compute_text` below before loading the tokenizer saved
by example 02. The compute callback uses a fixed count during inference so cached and
rebuilt histories contain identical text. Keep the compute text in the format callback
identical to training; setup verifies its exact token count. Set the count before
starting an episode.


In [ ]:
def episode_separator(ctx):
    ended = ctx.get("inner_terminated", False) or ctx.get("inner_truncated", False)
    return "\n\n" if ended else ""


def compute_text(ctx):
    if not ctx.get("action_requested", False):
        return ""
    return " ." * INFERENCE_COMPUTE_TOKENS


class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
            max_attention_distance_tokens=MAX_ATTENTION_DISTANCE_TOKENS,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"],
                            max_batch_tokens=DECODE_MAX_BATCH_TOKENS,
                            cache_capacity_tokens=DECODE_CACHE_CAPACITY_TOKENS,
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class QHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


tokenizer = load_tokenizer(repo_id_or_path=MODEL_ID, force_download=True)


# Check once at setup: each compute position must be one identical token.
compute_token_ids = tokenizer.tokenizers["text"].encode(" .", add_special_tokens=False)
if len(compute_token_ids) != 1:
    raise ValueError("Compute text must encode to exactly one token")
for compute_count in (INFERENCE_COMPUTE_TOKENS,):
    if (
        tokenizer.tokenizers["text"].encode(
            " ." * compute_count, add_special_tokens=False
        )
        != compute_token_ids * compute_count
    ):
        raise ValueError("Repeated compute text must preserve the exact token count")

encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)
q_head = (
    QHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

if not isinstance(tokenizer, Tokenizer):
    raise ValueError("Inference requires an Tokenizer checkpoint")
load_model(
    model=modules, repo_id_or_path=MODEL_ID, force_download=True, map_location=device
)
modules.eval()

## Run

The encoder uses Hugging Face's continuous-batching scheduler and paged KV allocator
with Mouse's existing embeddings, FlexAttention decoder, and Q head. Long inputs
are split at `DECODE_MAX_BATCH_TOKENS`; decoder layers process only real tokens.
The environment loop here remains lockstep. For independently ready environments,
Core also exposes `session.submit(row, embeds)` and `session.step()`.

`DECODE_CACHE_CAPACITY_TOKENS` bounds the shared pool; it does not change attention.
Training and inference both use `MAX_ATTENTION_DISTANCE_TOKENS`. Each query sees
self and up to that many previous tokens in its own outer episode. Set the distance
above the longest tokenized episode, including prompt and compute tokens, to retain
full-episode behavior. HF recycles slots in a circular KV buffer when the window
fills; rotary positions keep advancing. Inner resets retain that history. A new `episode_index` clears only its own
row through `cache.reset_rows`. There is no periodic rebuild; KV expires only when the shared mask excludes it.
If capacity is insufficient, decoding raises: increase the pool or reduce the
number of simultaneous environments. A full ring needs `distance + 1` KV slots
per stream, rounded up to a multiple of 16. The example uses two streams to limit the
fixed pool's memory requirement. Close the cache before changing weights.

Decode takes model inputs (`TensorDict`) from `Tokenizer.pack_rows`, using the same
tokenizer and sequence boundaries as training. Completed-episode scores come from
the trackers' completed task summaries.


In [ ]:
eval_contexts = [[] for _ in eval_env.names]


@torch.no_grad()
def run_eval(*, num_steps: int, temperature: float):
    """Greedy inference with frame capture; scores from ``trackers``.

    ``num_steps`` is lockstep ``GroupEnv`` steps for this call.
    Row ``contexts`` persist across the call (mutated in place). The KV cache
    starts from the full current episode, then consumes only new events.
    A new ``episode_index`` clears that stream's context and resets its KV row.
    Completed-episode scores come from ``trackers``.
    """
    modules.eval()
    n = len(eval_env.names)
    if len(eval_contexts) != n:
        raise ValueError(f"contexts has {len(eval_contexts)} streams but env has {n}")
    kv_cache = None
    cached_ends = np.zeros(n, dtype=np.int64)
    inputs = None
    outputs = None
    for tracker in trackers:
        tracker.clear()
    action_space = eval_env.action_space.spaces[0]
    assert isinstance(action_space, gym.spaces.Discrete)
    num_actions = int(action_space.n)
    video_names = eval_env.names
    video_envs = eval_env.envs
    frames_per_env = [[] for _ in video_names]
    steps_done = 0

    def _act_from_contexts() -> list[int | None]:
        nonlocal kv_cache, cached_ends
        ends = np.array([len(c) for c in eval_contexts], dtype=np.int64)
        batch = [
            eval_contexts[i][int(cached_ends[i]) : int(ends[i])]
            for i in range(n)
        ]
        with torch.no_grad():
            inputs, inputs_objective_data = tokenizer.pack_rows(
                rows=[[event for step in row for event in step] for row in batch],
            )
            out = encoder(inputs, cache=kv_cache, use_cache=True)
            kv_cache = out["cache"]
            cached_ends = ends.copy()
            requested = [
                i for i, tracker in enumerate(trackers) if tracker.action_requested
            ]
            actions = select_discrete_action(
                scores=action_readouts(
                    q_head(out["z"]), inputs, sequence_ids=requested
                ),
                temperature=temperature,
                num_actions=num_actions,
            )
        chosen = iter(actions.cpu().tolist())
        return [
            next(chosen) if tracker.action_requested else None for tracker in trackers
        ]

    try:
        for _ in range(num_steps):
            event_starts = [len(log) for log in logs]
            for log, tracker in zip(logs, trackers):
                episode_start = tracker.output is None or tracker.ended
                if episode_start:
                    log.event(
                        provider="coordinator",
                        episode_index=tracker.next_episode_index,
                        step_index=(0 if episode_start else tracker.step_index + 1),
                        payload={
                            "_episode_start": True,
                        },
                    )

            if any(tracker.action_requested for tracker in trackers):
                inputs = _act_from_contexts()
            else:
                inputs = [None] * n
            for log, tracker, action in zip(logs, trackers, inputs):
                if action is not None:
                    log.event(
                        provider="agent",
                        episode_index=tracker.next_episode_index,
                        step_index=tracker.step_index + 1,
                        payload={
                            "action": action,
                        },
                    )
            outputs = eval_env.step(tuple(inputs))
            for i, (action, out) in enumerate(zip(inputs, outputs)):
                log = logs[i]
                tracker = trackers[i]
                tracker.update(output=out)

                log.event(
                    provider="environment",
                    episode_index=tracker.episode_index,
                    step_index=tracker.step_index,
                    payload={
                        "observation": tracker.observation,
                        **(
                            {"repeat_episode_index": tracker.repeat_episode_index}
                            if tracker.repeat_episode_start
                            else {}
                        ),
                        **(
                            {"inner_terminated": True}
                            if tracker.inner_terminated
                            else {}
                        ),
                        **(
                            {"inner_truncated": True} if tracker.inner_truncated else {}
                        ),
                        **(
                            {"_q_star": tracker.info["q_star"]}
                            if tracker.action_requested and "q_star" in tracker.info
                            else {}
                        ),
                        **({"_terminated": True} if tracker.terminated else {}),
                        **({"_truncated": True} if tracker.truncated else {}),
                        **({"reward": tracker.reward} if tracker.reward != 0 else {}),
                        **(
                            {"action_requested": True}
                            if tracker.action_requested
                            else {}
                        ),
                    },
                )
                row = log[event_starts[i] :]
                if eval_contexts[i] and tracker.episode_start:
                    eval_contexts[i] = []
                    cached_ends[i] = 0
                    if kv_cache is not None:
                        kv_cache.reset_rows(rows=[i])
                eval_contexts[i].append(row)
            for i, video_env in enumerate(video_envs):
                if eval_contexts[i][-1][-1]["episode_index"] >= EPISODES_PER_ENV:
                    continue
                frame = video_env.render()
                if frame is not None:
                    frames_per_env[i].append(np.asarray(frame).copy())
            steps_done += 1
            if steps_done % 100 == 0:
                episode_scores = [
                    task.reward for tracker in trackers for task in tracker.tasks
                ]
                mean_episode_score = (
                    sum(episode_scores) / len(episode_scores)
                    if episode_scores
                    else float("nan")
                )
                print(
                    f"step {steps_done} | {len(episode_scores)} episodes this window | mean episode score {mean_episode_score:.2f}/{REPEAT_COUNT}",
                )
    finally:
        if kv_cache is not None:
            kv_cache.close()
    scores_per_env = [[task.reward for task in tracker.tasks] for tracker in trackers]
    return (video_names, frames_per_env, scores_per_env)


video_names, frames_per_env, eval_episode_scores = run_eval(
    num_steps=EVAL_STEPS, temperature=0.0
)
eval_env.close()

## Replay MP4s

The remaining cells format environment-specific render frames as MP4s. This is not
required for Mouse Core inference, but it is a useful way to inspect what the loaded
policy is doing.


In [ ]:
def format_episode_scores(*, episode_scores):
    episode_scores = episode_scores[:EPISODES_PER_ENV]
    scores_text = " ".join((f"{score:.0f}" for score in episode_scores))
    summary = f"episode scores [{scores_text}] / {REPEAT_COUNT} | mean {sum(episode_scores) / EPISODES_PER_ENV:.2f}"
    return summary


def display_env_replay(*, name, frames, episode_scores, fps=5):
    print(f"{name}: {format_episode_scores(episode_scores=episode_scores)}")
    if not frames:
        print("  (no frames captured)")
        return
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(w / 100, h / 100))
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


for name, frames, episode_scores in zip(
    video_names, frames_per_env, eval_episode_scores
):
    display_env_replay(name=name, frames=frames, episode_scores=episode_scores)